In [1]:
%%writefile ../src/business.py
"""Cost-based decision layer.

Framing: flagging an order costs c_int (an intervention: upgrade, expedite,
proactive notice). An intervention prevents lateness with probability
`effectiveness`. A late order that is not prevented costs c_late (SLA penalty,
unhappy customer). These are ASSUMPTIONS, not figures from the data.

Flag an order when  effectiveness * P(late) * c_late > c_int,
i.e. when P(late) > c_int / (effectiveness * c_late).
"""
import numpy as np
import pandas as pd


def break_even_probability(cost_ratio: float, effectiveness: float = 1.0) -> float:
    """cost_ratio = c_late / c_int."""
    return 1.0 / (cost_ratio * effectiveness)


def policy_result(y, flagged, cost_ratio: float, effectiveness: float = 1.0) -> dict:
    """Net saving versus doing nothing, in units of one intervention cost,
    per 1,000 orders, using the real outcomes y (1 = late)."""
    y = np.asarray(y); flagged = np.asarray(flagged).astype(bool)
    saving = (effectiveness * y[flagged] * cost_ratio - 1.0).sum()
    return {
        "share_flagged": flagged.mean(),
        "recall": (y[flagged].sum() / y.sum()) if y.sum() else np.nan,
        "precision": y[flagged].mean() if flagged.any() else np.nan,
        "saving_per_1000": 1000 * saving / len(y),
    }


def policy_grid(y, p, ratios=(2, 3, 5, 10), efficacies=(0.3, 0.5, 1.0)) -> pd.DataFrame:
    """Best flagging rule for each (cost ratio, effectiveness) pair."""
    p = np.asarray(p)
    rows = []
    for r in ratios:
        for e in efficacies:
            t = break_even_probability(r, e)
            res = policy_result(y, p > t, r, e)
            all_res = policy_result(y, np.ones(len(p), bool), r, e)
            rows.append({"cost_ratio": r, "effectiveness": e,
                         "threshold": round(t, 3), **res,
                         "saving_flag_all": all_res["saving_per_1000"]})
    return pd.DataFrame(rows)

Writing ../src/business.py


In [2]:
import sys
sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from src import config
from src.features import split_feature_types
from src.models import make_logreg, make_hgb
from src.evaluate import oof_predictions
from src.business import policy_grid

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

train = pd.read_parquet(config.ROOT / "data/processed/train.parquet")
y, groups = train[config.TARGET], train["Customer Id"]
numeric, categorical = split_feature_types(train)
MODE = [config.SHIP_MODE]
all_feats = numeric + categorical

p_mode = oof_predictions(make_logreg(MODE, numeric), train[MODE], y, groups)
print(len(p_mode), "out-of-fold predictions")

138213 out-of-fold predictions


In [3]:
cal = (train.assign(p=p_mode).groupby(config.SHIP_MODE)
       .agg(orders=("p", "size"), mean_predicted=("p", "mean"),
            observed_late=(config.TARGET, "mean")).round(4))
print(cal)
print("\nOverall predicted vs observed:", round(p_mode.mean(), 4), round(y.mean(), 4))

                orders  mean_predicted  observed_late
Shipping Mode                                        
First Class      21217          0.9996         1.0000
Same Day          7488          0.4818         0.4813
Second Class     27144          0.7947         0.7947
Standard Class   82364          0.3982         0.3981

Overall predicted vs observed: 0.5729 0.5729


In [4]:
p_hgb = oof_predictions(
    make_hgb(all_feats, numeric, max_depth=3, learning_rate=0.05,
             max_iter=200, early_stopping=False),
    train[all_feats], y, groups)

decile = pd.qcut(pd.Series(p_hgb).rank(method="first"), 10, labels=False) + 1
(pd.DataFrame({"pred": p_hgb, "obs": y.values, "decile": decile})
   .groupby("decile").agg(mean_predicted=("pred", "mean"), observed_late=("obs", "mean"))
   .round(3))

,mean_predicted,observed_late
decile,,
1,0.340,0.397
2,0.384,0.404
3,0.397,0.401
4,0.405,0.396
5,0.415,0.401
6,0.445,0.412
7,0.632,0.633
8,0.790,0.795
9,0.922,0.890


In [5]:
grid = policy_grid(y, p_mode)
grid.round(3)

,cost_ratio,effectiveness,threshold,share_flagged,recall,precision,saving_per_1000,saving_flag_all
0,2,0.3,1.667,0.000,0.000,NaN,0.000,-656.261
1,2,0.5,1.000,0.000,0.000,NaN,0.000,-427.102
2,2,1.0,0.500,0.350,0.540,0.885,269.244,145.797
3,3,0.3,1.111,0.000,0.000,NaN,0.000,-484.391
4,3,0.5,0.667,0.350,0.540,0.885,114.457,-140.652
5,3,1.0,0.333,1.000,1.000,0.573,718.695,718.695
6,5,0.3,0.667,0.350,0.540,0.885,114.457,-140.652
7,5,0.5,0.400,0.404,0.586,0.831,435.042,432.246
8,5,1.0,0.200,1.000,1.000,0.573,1864.492,1864.492
9,10,0.3,0.333,1.000,1.000,0.573,718.695,718.695


In [6]:
need = (1 / pd.Series(p_mode).groupby(train[config.SHIP_MODE].values).mean()).round(2)
need.sort_values().rename("flag this mode when cost_ratio x effectiveness is above")

First Class       1.00
Second Class      1.26
Same Day          2.08
Standard Class    2.51
Name: flag this mode when cost_ratio x effectiveness is above, dtype: float64

In [7]:
rows = []
for mode, g in train.groupby(config.SHIP_MODE):
    row = {"mode": mode, "scheduled_now": int(g[config.SCHED_DAYS].iloc[0]),
           "late_now": round(g[config.TARGET].mean(), 3)}
    for tgt in (0.30, 0.20, 0.10):
        row[f"promise_for_<={int(tgt*100)}%_late"] = next(
            s for s in range(8) if (g[config.REAL_DAYS] > s).mean() <= tgt)
    rows.append(row)
pd.DataFrame(rows).set_index("mode")

,scheduled_now,late_now,promise_for_<=30%_late,promise_for_<=20%_late,promise_for_<=10%_late
mode,,,,,
First Class,1,1.000,2,2,2
Same Day,0,0.481,1,1,1
Second Class,2,0.795,5,5,6
Standard Class,4,0.398,5,6,6
